In [ ]:
# ── All Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# DS 612 — Group Project: Used Car Price Prediction

**Authors:** Ana Sachuk · Terry · Nadine · Aarnav

**Dataset:** Craigslist Used Vehicle Listings (`vehicles.csv`) — 426,880 records

**Research Question:** Can we accurately predict the listing price of a used car from its attributes?

---

| Section | Author | Content |
|---|---|---|
| 1 | Ana | Data Cleaning + Baseline Models (Linear, Decision Tree, Ridge) |
| 2 | Terry | Polynomial Regression, Tuned Random Forest + Visualizations |
| 3 | Nadine | Random Forest + MLR + Model Comparison |
| 4 | Aarnav | Gradient Boosting + Lasso Regression |

---
# Section 1 — Ana's Work
*(from `Datacleaning.ipynb` and `ana_model.ipynb`)*

In [ ]:
vehiclesdf_unclean = pd.read_csv(r'C:\Users\arnie\Coding\DS_612_GROUP\vehicles.csv')

vehiclesdfc = vehiclesdf_unclean[["id","region","region_url","price","year","manufacturer","model","condition","cylinders","state","posting_date"]]

In [ ]:
vehiclesdfc.head()

In [ ]:
print(vehiclesdfc.shape)
print(" ")
print(vehiclesdfc.info())
print(vehiclesdfc.describe())
print((vehiclesdfc.isnull().sum() / len(vehiclesdfc)) * 100)

In [ ]:
# Due to condition and cylinders being at 40% with nAN values then we should drop it 
vehiclesdfc = vehiclesdfc.drop(columns=["condition", "cylinders"])
vehiclesdfc = vehiclesdfc.dropna()

In [ ]:
vehiclesdfc.head()

In [ ]:
# Filter out extreme outliers first
df_plot = vehiclesdfc[vehiclesdfc['price'].between(500, 80000)]

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df_plot['price'], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)

ax.set_title('Vehicle price distribution', fontsize=14)
ax.set_xlabel('Price ($)')
ax.set_ylabel('Number of listings')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x):,}'))

plt.tight_layout()
plt.show()

In [ ]:
# clean out bad price values FIRST
vehiclesdfc = vehiclesdfc[vehiclesdfc['price'] > 100]
vehiclesdfc = vehiclesdfc[vehiclesdfc['price'] < 100000]

# THEN plot
plt.hist(vehiclesdfc['price'], bins=50)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# selecting features for modeling
df_model = vehiclesdfc[['price', 'year']]
df_model = df_model.dropna()

In [ ]:
# X = features, y = target
X = df_model[['year']]
y = df_model['price']

In [ ]:
# splitting data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Model 1: Linear Regression
model1 = LinearRegression()
model1.fit(X_train, y_train)

y_pred1 = model1.predict(X_test)

mse1 = mean_squared_error(y_test, y_pred1)
print("Linear Regression MSE:", mse1)

In [ ]:
# Model 2: Decision Tree
model2 = DecisionTreeRegressor(max_depth=5)
model2.fit(X_train, y_train)

y_pred2 = model2.predict(X_test)

mse2 = mean_squared_error(y_test, y_pred2)
print("Decision Tree MSE:", mse2)

In [ ]:
# Model 3: Ridge Regression
model3 = Ridge(alpha=1.0)
model3.fit(X_train, y_train)

y_pred3 = model3.predict(X_test)

mse3 = mean_squared_error(y_test, y_pred3)
print("Ridge MSE:", mse3)

---
# Section 2 — Terry's Work
*(from `Terry_model_testing.ipynb`)*

#This is the clean data set 

In [ ]:
# Use 'on_bad_lines' to skip the broken row and 'engine' to handle messy quotes
vehiclesdf_unclean = pd.read_csv(r'C:\Users\arnie\Coding\DS_612_GROUP\vehicles.csv', on_bad_lines='skip', engine='python')

# Now you can filter your columns as planned
vehiclesdfc = vehiclesdf_unclean[["id", "region", "region_url", "price", "year", "manufacturer", "model", "condition", "cylinders", "state", "posting_date"]]

# Check to see if it loaded
vehiclesdfc.head()

In [ ]:
vehiclesdfc.head()

In [ ]:
print(vehiclesdfc.shape)
print(" ")
print(vehiclesdfc.info())
print(vehiclesdfc.describe())
print((vehiclesdfc.isnull().sum() / len(vehiclesdfc)) * 100)

In [ ]:
# Due to condition and cylinders being at 40% with nAN values then we should drop it 
vehiclesdfc = vehiclesdfc.drop(columns=["condition", "cylinders"])
vehiclesdfc = vehiclesdfc.dropna()

In [ ]:
vehiclesdfc.head()

# Next are models: Multiple Linear Regression, Polynomial Regression, and Random Forest regression

In [ ]:
# 1. Drop columns that won't help predict price or are too messy (like URLs and IDs)
columns_to_drop = ['id', 'region', 'region_url', 'model', 'state', 'posting_date']
ml_df = vehiclesdfc.drop(columns=columns_to_drop)

# 2. Convert remaining text (like 'manufacturer') into 1s and 0s
ml_df = pd.get_dummies(ml_df, drop_first=True)

# 3. Separate Features (X) and Target (y)
X = ml_df.drop('price', axis=1)
y = ml_df['price']

# 4. Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data prepared! Training on {len(X_train)} samples.")

In [ ]:
# 1. Filter out the "trash" prices and years
# This removes the $0 and $1 entries that are breaking your math
vehicles_filtered = vehiclesdfc[(vehiclesdfc['price'] > 500) & (vehiclesdfc['price'] < 150000)]
vehicles_filtered = vehicles_filtered[(vehicles_filtered['year'] > 1990) & (vehicles_filtered['year'] < 2025)]

# 2. Re-run the preparation
columns_to_drop = ['id', 'region', 'region_url', 'model', 'state', 'posting_date']
ml_df = vehicles_filtered.drop(columns=columns_to_drop)
ml_df = pd.get_dummies(ml_df, drop_first=True)

X = ml_df.drop('price', axis=1)
y = ml_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Now run your models - you should see positive R-squared values!

In [ ]:
# 1. Create the Transformer (degree=2 creates Year^2)
poly = PolynomialFeatures(degree=2)

# 2. Transform your original X data into Polynomial data
# This creates the 'X_train_poly' and 'X_test_poly' variables the model is looking for
X_train_poly = poly.fit_transform(X_train[['year']])
X_test_poly = poly.transform(X_test[['year']])

# 3. NOW your existing code will work:
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

In [ ]:
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)
print(f"Polynomial Regression (Year Only) R-squared: {r2_score(y_test, y_pred_poly):.4f}")

# I should've scored higher with this model, but due to a lack of information, it scored low 

# Random Forest Regressor is the best model to use out of the three

In [ ]:
# --- Tuned Random Forest Regressor ---
# n_estimators: Number of trees (more is usually better but slower)
# max_depth: Limits how complex each tree gets to prevent overfitting
# n_jobs=-1: Uses all your computer's processors to speed it up

rf_tuned = RandomForestRegressor(
    n_estimators=300, 
    max_depth=15, 
    min_samples_split=5, 
    random_state=42, 
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)
y_pred_tuned = rf_tuned.predict(X_test)

print(f"Original Random Forest R-squared: 0.5724")
print(f"Tuned Random Forest R-squared: {r2_score(y_test, y_pred_tuned):.4f}")

# set max_depth to 15, which focuses on the general picture and getting rid of outliers that don't represent the whole market 

In [ ]:
df=vehiclesdfc.copy()

In [ ]:
# 1. Start fresh from your original data
df = vehiclesdfc.copy()
df.columns = [col.lower().strip() for col in df.columns]

# 2. Filter outliers (The "Clean-Up")
# We remove ultra-cheap cars (usually scams/parts) and ultra-expensive ones
df = df[(df['price'] > 1000) & (df['price'] < 80000)]

# 3. Feature Engineering
df['age'] = 2026 - df['year']

# 4. Use Manufacturer and State (One-Hot Encoding)
# We select the columns that actually impact price
features = ['age', 'manufacturer', 'state']
X = df[features].dropna()
y = df.loc[X.index, 'price'] # Match y to the rows we kept in X

# Convert text (Ford, Tesla, NY, CA) into numbers the model can read
X_encoded = pd.get_dummies(X, columns=['manufacturer', 'state'], drop_first=True)

# 5. Split and Train
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

rf_boosted = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf_boosted.fit(X_train, y_train)

# 6. Check the new score
predictions = rf_boosted.predict(X_test)
print(f"New R-squared : {r2_score(y_test, predictions):.4f}")

In [ ]:
# 1. SETUP: Prepare the data
df_viz = vehiclesdfc.copy()
df_viz.columns = [col.lower().strip() for col in df_viz.columns]
df_viz = df_viz[(df_viz['price'] > 1000) & (df_viz['price'] < 80000)].dropna(subset=['price', 'year'])
df_viz['age'] = 2026 - df_viz['year']

In [ ]:
# --- PLOT 1: Price Distribution ---
plt.figure(figsize=(10, 6))
sns.histplot(df_viz['price'], kde=True, color='royalblue')
plt.title('1. Distribution of Car Prices', fontsize=15)
plt.xlabel('Price ($)')
plt.show()

In [ ]:
#PLOT 2: Depreciation Curve 
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_viz, x='year', y='price', color='forestgreen')
plt.title('2. Average Price Trend by Year (Depreciation)', fontsize=15)
plt.xlabel('Model Year')
plt.show()

In [ ]:
# --- PLOT 3: Top 10 Brands ---
plt.figure(figsize=(12, 6))
top_brands = df_viz['manufacturer'].value_counts().nlargest(10).index
sns.boxplot(data=df_viz[df_viz['manufacturer'].isin(top_brands)], x='manufacturer', y='price', hue='manufacturer', palette='Set2', legend=False)
plt.title('3. Price Ranges for Top 10 Brands', fontsize=15)
plt.xticks(rotation=45)
plt.show()

---
# Section 3 — Nadine's Work
*(from `nadine_model.ipynb`)*

## Data Cleaning

In [ ]:
vehiclesdf_unclean = pd.read_csv(r'C:\Users\arnie\Coding\DS_612_GROUP\vehicles.csv')

vehiclesdfc = vehiclesdf_unclean[["id","region","region_url","price","year","manufacturer","model","condition","cylinders","state","posting_date"]]

In [ ]:
vehiclesdfc.head()

In [ ]:
print(vehiclesdfc.shape)
print(" ")
print(vehiclesdfc.info())
print(vehiclesdfc.describe())
print((vehiclesdfc.isnull().sum() / len(vehiclesdfc)) * 100)

In [ ]:
# Due to condition and cylinders being at 40% with nAN values then we should drop it 
vehiclesdfc = vehiclesdfc.drop(columns=["condition", "cylinders"])
vehiclesdfc = vehiclesdfc.dropna()

In [ ]:
vehiclesdfc.head()

## Preprocessing

In [ ]:
df = vehiclesdfc.copy()
df = df[(df["price"] >= 500) & (df["price"] <= 150_000)]
df["year"] = df["year"].astype(int)

print(f"After outlier removal: {df.shape}")
df["price"].describe()

## Random Forest

**Feature engineering** 

In [ ]:
df_rf = df.copy()

# Limit OHE to top 200 models — freeform Craigslist entries create 22,000+ unique
# model names, which makes training extremely slow. Keeping the top 200 covers the
# vast majority of listings while keeping the feature matrix manageable.
top_models = df_rf["model"].value_counts().nlargest(200).index
df_rf["model_grouped"] = df_rf["model"].where(df_rf["model"].isin(top_models), other="other")
model_ohe = pd.get_dummies(df_rf["model_grouped"], prefix="model", drop_first=True)

# Label encode the remaining low-cardinality categoricals
for col in ["manufacturer", "region", "state"]:
    le = LabelEncoder()
    df_rf[col + "_encoded"] = le.fit_transform(df_rf[col].astype(str))

# Combine all features
base_features = ["year", "manufacturer_encoded", "region_encoded", "state_encoded"]
X_rf = pd.concat([df_rf[base_features], model_ohe], axis=1)
y_rf = df_rf["price"]

print(f"X shape: {X_rf.shape}  (base features + {model_ohe.shape[1]} model dummies)")
print(f"y shape: {y_rf.shape}")

**Train/Test Split**

In [ ]:
X_rf_train, X_rf_test, y_rf_train, y_rf_test = train_test_split(X_rf, y_rf, test_size=0.2, random_state=0)

**Train model**

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=5, n_jobs=-1,random_state=0)
rf_model.fit(X_rf_train, y_rf_train)

**Evaluate**

In [ ]:
y_rf_pred = rf_model.predict(X_rf_test)

mae  = mean_absolute_error(y_rf_test, y_rf_pred)
rmse = np.sqrt(mean_squared_error(y_rf_test, y_rf_pred))
r2   = r2_score(y_rf_test, y_rf_pred)

print(f"MAE :  ${mae:,.0f}")
print(f"RMSE:  ${rmse:,.0f}")
print(f"R\u00b2  :  {r2:.3f}")

In [ ]:
# aliases so the comparison table below can reference rf_mae, rf_rmse, rf_r2
rf_mae, rf_rmse, rf_r2 = mae, rmse, r2

**Visualization**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#Top 15 most important features (OHE creates too many to show all)
importances = pd.Series(rf_model.feature_importances_, index=X_rf.columns)
importances.nlargest(15).sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Top 15 Feature Importances")
axes[0].set_xlabel("Importance Score")

#Actual vs Predicted
axes[1].scatter(y_rf_test, y_rf_pred, alpha=0.2, s=5, color="steelblue")
axes[1].plot([0, 150_000], [0, 150_000], "r--", linewidth=1.5, label="Perfect")
axes[1].set_xlabel("Actual Price ($)")
axes[1].set_ylabel("Predicted Price ($)")
axes[1].set_title(f"Actual vs Predicted  —  R\u00b2 = {rf_r2:.3f}")
axes[1].legend()

plt.suptitle("Random Forest Regressor", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Multiple Linear

**Train/Test Split**

In [ ]:
X_mlr_train, X_mlr_test = X_rf_train, X_rf_test
y_mlr_train, y_mlr_test = y_rf_train, y_rf_test

**Train model**

In [ ]:
mlr_model = LinearRegression(n_jobs=-1)
mlr_model.fit(X_mlr_train, y_mlr_train)

**Evaluation**

In [ ]:
y_mlr_pred = mlr_model.predict(X_mlr_test)

mlr_mae  = mean_absolute_error(y_mlr_test, y_mlr_pred)
mlr_rmse = np.sqrt(mean_squared_error(y_mlr_test, y_mlr_pred))
mlr_r2   = r2_score(y_mlr_test, y_mlr_pred)

print(f"MAE :  ${mlr_mae:,.0f}")
print(f"RMSE:  ${mlr_rmse:,.0f}")
print(f"R\u00b2  :  {mlr_r2:.3f}")

**Visualization**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#Actual vs Predicted
axes[0].scatter(y_mlr_test, y_mlr_pred, alpha=0.2, s=5, color="darkorange")
axes[0].plot([0, 150_000], [0, 150_000], "r--", linewidth=1.5, label="Perfect")
axes[0].set_xlabel("Actual Price ($)")
axes[0].set_ylabel("Predicted Price ($)")
axes[0].set_title(f"Actual vs Predicted  —  R\u00b2 = {mlr_r2:.3f}")
axes[0].legend()

#Top 15 largest absolute coefficients (base features only for clarity)
coef_series = pd.Series(mlr_model.coef_, index=X_rf.columns)
coef_series[base_features].plot(
    kind="barh", ax=axes[1],
    color=["steelblue" if v >= 0 else "tomato" for v in coef_series[base_features]]
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Coefficients — Base Features")
axes[1].set_xlabel("Effect on Price ($)")

plt.suptitle("Multiple Linear Regression", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Multiple Linear Regression"],
    "MAE ($)": [f"{rf_mae:,.0f}", f"{mlr_mae:,.0f}"],
    "RMSE ($)": [f"{rf_rmse:,.0f}", f"{mlr_rmse:,.0f}"],
    "R\u00b2": [f"{rf_r2:.3f}", f"{mlr_r2:.3f}"]
})
print(comparison.to_string(index=False))

---
# Section 4 — Aarnav's Work

Two models not covered by the rest of the group:

| Model | Why it's worth trying |
|---|---|
| **Gradient Boosting Regressor** | Builds trees **sequentially** — each tree corrects the previous one's errors. Typically outperforms Random Forest (bagging) on structured tabular data by focusing on hard-to-predict examples. |
| **Lasso Regression** | L1 regularization that shrinks unimportant feature coefficients to exactly **zero**, effectively doing automatic feature selection across our 200+ features. Shows which features genuinely matter. |

In [ ]:
# Load and clean
vehiclesdf_unclean = pd.read_csv(r'C:\Users\arnie\Coding\DS_612_GROUP\vehicles.csv')
vehiclesdfc = vehiclesdf_unclean[["id","region","region_url","price","year","manufacturer","model","condition","cylinders","state","posting_date"]]
vehiclesdfc = vehiclesdfc.drop(columns=["condition", "cylinders"])
vehiclesdfc = vehiclesdfc.dropna()

df_aarnav = vehiclesdfc.copy()
df_aarnav = df_aarnav[(df_aarnav["price"] >= 500) & (df_aarnav["price"] <= 150_000)]
df_aarnav["year"] = df_aarnav["year"].astype(int)

print(f"Dataset ready: {df_aarnav.shape[0]:,} rows")

In [ ]:
df_a = df_aarnav.copy()

top_models_a = df_a["model"].value_counts().nlargest(200).index
df_a["model_grouped"] = df_a["model"].where(df_a["model"].isin(top_models_a), other="other")
model_ohe_a = pd.get_dummies(df_a["model_grouped"], prefix="model", drop_first=True)

for col in ["manufacturer", "region", "state"]:
    le = LabelEncoder()
    df_a[col + "_encoded"] = le.fit_transform(df_a[col].astype(str))

aarnav_base_features = ["year", "manufacturer_encoded", "region_encoded", "state_encoded"]
X_aarnav = pd.concat([df_a[aarnav_base_features], model_ohe_a], axis=1)
y_aarnav = df_a["price"]

print(f"Feature matrix: {X_aarnav.shape[0]:,} rows × {X_aarnav.shape[1]} features")

In [ ]:
X_aarnav_train, X_aarnav_test, y_aarnav_train, y_aarnav_test = train_test_split(
    X_aarnav, y_aarnav, test_size=0.2, random_state=42
)
print(f"Train: {X_aarnav_train.shape[0]:,} | Test: {X_aarnav_test.shape[0]:,} | Features: {X_aarnav_train.shape[1]}")

## Gradient Boosting Regressor

Unlike Random Forest which builds trees in **parallel** (bagging), Gradient Boosting builds trees **sequentially** — each new tree is trained to correct the residual errors left by the previous ensemble. This focuses the model on the hardest-to-predict examples and typically achieves higher accuracy.

Using `HistGradientBoostingRegressor` (sklearn's fast histogram-based implementation, similar to LightGBM):

- `max_iter=200`: number of boosting stages
- `max_depth=6`: shallow trees to avoid overfitting  
- `learning_rate=0.1`: step size for each correction

In [ ]:
print("Training Gradient Boosting Regressor ...")
gb_model = HistGradientBoostingRegressor(
    max_iter=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)
gb_model.fit(X_aarnav_train, y_aarnav_train)
print("Done.")

In [ ]:
y_gb_pred = gb_model.predict(X_aarnav_test)

gb_mae  = mean_absolute_error(y_aarnav_test, y_gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_aarnav_test, y_gb_pred))
gb_r2   = r2_score(y_aarnav_test, y_gb_pred)

print(f"MAE :  ${gb_mae:,.0f}")
print(f"RMSE:  ${gb_rmse:,.0f}")
print(f"R²  :  {gb_r2:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 15 feature importances
gb_importances = pd.Series(gb_model.feature_importances_, index=X_aarnav.columns)
gb_importances.nlargest(15).sort_values().plot(kind="barh", ax=axes[0], color="mediumseagreen")
axes[0].set_title("Top 15 Feature Importances — Gradient Boosting")
axes[0].set_xlabel("Importance Score")

# Actual vs Predicted (sample 3000 points for readability)
sample_idx = np.random.choice(len(y_aarnav_test), size=3000, replace=False)
y_test_arr = np.array(y_aarnav_test)
axes[1].scatter(y_test_arr[sample_idx], y_gb_pred[sample_idx], alpha=0.2, s=5, color="mediumseagreen")
axes[1].plot([0, 150_000], [0, 150_000], "r--", linewidth=1.5, label="Perfect")
axes[1].set_xlabel("Actual Price ($)")
axes[1].set_ylabel("Predicted Price ($)")
axes[1].set_title(f"Actual vs Predicted  —  R² = {gb_r2:.3f}")
axes[1].legend()

plt.suptitle("Gradient Boosting Regressor", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Lasso Regression

Lasso adds an **L1 penalty** to linear regression: it minimizes prediction error *plus* the sum of absolute coefficient values. This forces unimportant feature weights all the way to **zero** — acting as automatic feature selection across our 200+ features.

- `alpha=100`: regularization strength (higher = more coefficients zeroed out)
- `max_iter=3000`: enough iterations for convergence on a large feature matrix

In [ ]:
lasso_model = Lasso(alpha=100, max_iter=3000)
lasso_model.fit(X_aarnav_train, y_aarnav_train)
print("Lasso trained.")

In [ ]:
y_lasso_pred = lasso_model.predict(X_aarnav_test)

lasso_mae  = mean_absolute_error(y_aarnav_test, y_lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_aarnav_test, y_lasso_pred))
lasso_r2   = r2_score(y_aarnav_test, y_lasso_pred)

print(f"MAE :  ${lasso_mae:,.0f}")
print(f"RMSE:  ${lasso_rmse:,.0f}")
print(f"R²  :  {lasso_r2:.3f}")
print(f"\nLasso zeroed out {(lasso_model.coef_ == 0).sum()} of {X_aarnav.shape[1]} features ({(lasso_model.coef_ == 0).mean():.1%})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Non-zero coefficients for base features
coef_lasso = pd.Series(lasso_model.coef_, index=X_aarnav.columns)
lasso_base_coef = coef_lasso[aarnav_base_features].sort_values()
lasso_base_coef.plot(
    kind="barh", ax=axes[0],
    color=["steelblue" if v >= 0 else "tomato" for v in lasso_base_coef]
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Lasso Coefficients — Base Features")
axes[0].set_xlabel("Effect on Price ($)")

# Actual vs Predicted
axes[1].scatter(y_test_arr[sample_idx], y_lasso_pred[sample_idx], alpha=0.2, s=5, color="darkorchid")
axes[1].plot([0, 150_000], [0, 150_000], "r--", linewidth=1.5, label="Perfect")
axes[1].set_xlabel("Actual Price ($)")
axes[1].set_ylabel("Predicted Price ($)")
axes[1].set_title(f"Actual vs Predicted  —  R² = {lasso_r2:.3f}")
axes[1].legend()

plt.suptitle("Lasso Regression", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
aarnav_comparison = pd.DataFrame({
    "Model": ["Gradient Boosting", "Lasso Regression"],
    "MAE ($)": [f"{gb_mae:,.0f}", f"{lasso_mae:,.0f}"],
    "RMSE ($)": [f"{gb_rmse:,.0f}", f"{lasso_rmse:,.0f}"],
    "R²": [f"{gb_r2:.3f}", f"{lasso_r2:.3f}"]
})
print(aarnav_comparison.to_string(index=False))